In [1]:
import os
import json
import numpy as np

In [2]:
profession_folders = os.listdir()
to_convert = {folder:folder+'/' for folder in profession_folders if (
    os.path.isdir(folder) and folder != 'alchemy')} #alchemy was already done manually so it doesn't have the CSVs

In [3]:
def comma_to_pipe(s):
    # CSV Standards -> fields separated by a ,
    #               -> any field containing , or a " or a \n must be surrounded be encased in " "
    #               -> a " in the field is escaped using another "
    # ergo, first quote of a field must be an encasing quote and all even sets of quotes are escaped versions, and all
    # odd sets of quotes must have an opening or closing encasing quote as part of the group
    new_s = ''
    open_quote = False
    is_escaped = False
    for l in s:
        if l == '"' and not open_quote:
            # current character is a quote, and we're not in a quotation set -> has to be opening a quotation set
            # by CSV standards (a field containing a " must be encased in "s)
            open_quote = True
            is_escaped = False
            continue
        elif l == ',' and not open_quote:
            # current character is a , and we're outside a quote set -> must be a delimiter -> swap to |
            # and move on to next character
            new_s += '|'
            continue
        elif l == ',' and is_escaped:
            # previous character was a potential escaping quote but current is not a quote -> previous must have been
            # a closing quote instead
            new_s = new_s[:-1] + '|'
        elif l == '"' and is_escaped:
            # current character is a quote and was escaped by previous character -> skip it because we printed the
            # previous quote
            is_escaped = False
            continue
        elif l == '"' and not is_escaped:
            # current character is a quote and was not escaped by previous character -> could be an escaping " and a
            # closing ", we don't know yet until we set next character, so pre-emptively set the is_escaped flag
            is_escaped = True
        elif l != '"' and is_escaped:
            # current character is not a quote, but the prevous character was a potential escaping quote -> the previous
            # must have been a closing " instead.
            is_escaped = False
            open_quote = False
            new_s = new_s[:-1]
            
        new_s += l
            
    return new_s

In [4]:
def preprocess_row(row):
    row = row.strip()
    row = comma_to_pipe(row)    
    row = row.split('|')
    
    last_field = np.where(np.array(row) != '')[0].max()
    row = row[:last_field+1]
    
    return row

In [5]:
def recipes_to_json(profession):
    file_path = to_convert.get(profession) + profession + '_recipes.csv'
    with open(file_path, 'r') as fin:
        data = fin.readlines()[1:]
        all_recipes = list()
        for recipe in data:
            recipe = preprocess_row(recipe)
            
            recipe_dict = dict()
            recipe_dict['itemID'] = recipe[0]
            recipe_dict['itemName'] = recipe[1]
            recipe_dict['reagents'] = dict(zip(recipe[2::2], [int(val) for val in recipe[3::2]]))
            
            all_recipes.append(recipe_dict)
    
    new_file_path = file_path[:-4] + '.json'
    with open(new_file_path, 'w') as fout:
        json.dump(all_recipes, fout, indent=4)

In [6]:
def specs_to_json(profession):
    file_path = to_convert.get(profession) + profession + '_specializations.csv'
    with open(file_path, 'r') as fin:
        data = fin.readlines()[1:]
        all_items = list()
        for item in data:
            item = preprocess_row(item)
            
            item_dict = dict()
            item_dict['itemID'] = item[0]
            item_dict['itemName'] = item[1]
            item_dict['nodes'] = item[2:]
            
            all_items.append(item_dict)
            
    new_file_path = file_path[:-4] + '.json'
    with open(new_file_path, 'w') as fout:
        json.dump(all_items, fout, indent=4)

In [7]:
def subnode_to_dict(profession, subnode):
    file_path = to_convert.get(profession) + profession + f'_{subnode}.csv'
    with open(file_path, 'r') as fin:
        header = fin.readline()
        header = preprocess_row(header)
        data = fin.readlines()
        all_subnodes = dict()
        for subnode in data:
            subnode = preprocess_row(subnode)
            all_subnodes[subnode[0]] = dict(zip(header[1:], [int(val) for val in subnode[1:]])) 
            
    return all_subnodes

In [8]:
def nodes_to_json(profession):
    subnodes = ['skill', 'ingenuity', 'resourcefulness', 'multicraft']
    nodes_dict = dict(zip(subnodes, [subnode_to_dict(profession, subnode) for subnode in subnodes]))
    
    new_file_path = to_convert.get(profession) + profession + '_nodes.json'
    with open(new_file_path, 'w') as fout:
        json.dump(nodes_dict, fout, indent=4)

In [9]:
professions = [
    'blacksmithing', 'cooking', 'enchanting', 'engineering', 'inscription', 'jewelcrafting', 'leatherworking', 'tailoring'
]

for profession in professions:
    print(f'current profession: {profession}')
    recipes_to_json(profession)
    
    if profession != 'cooking':
        specs_to_json(profession)
        nodes_to_json(profession)

current profession: blacksmithing
current profession: cooking
current profession: enchanting
current profession: engineering
current profession: inscription
current profession: jewelcrafting
current profession: leatherworking
current profession: tailoring
